### 特征获取

In [ ]:
from ta import add_all_ta_features
import numpy as np
import pandas as pd
from sklearn import preprocessing
from keras.models import Model
from keras.layers import Input
from keras import regularizers, optimizers
from keras.layers import Dense, Activation
import os
import config

In [ ]:
#数据降维方法：协方差代表的是金融资产或投资组合中不同资产收益率之间的同步波动性。
def add_cov_matrix(df):
    # 日期和时间戳对数据进行排序，并重置索引
    df=df.sort_values(['date','tic'],ignore_index=True) 
    # 将日期列转换为分类数据，并使用其编码值作为索引
    df.index = df.date.factorize()[0]
    # create empty list for storing coveriance matrices at each time step
    ## 设置构建协方差矩阵时回溯的时间长度为一年，这里假设一年有252个交易日
    cov_list = [] 
    # look back for constructing the coveriance matrix is one year
    lookback=251
    count=0
    count_list = []
    for i in range(lookback,len(df.index.unique())):
        # 获取当前时间点向前回溯lookback长度的数据
        data_lookback = df.loc[i-lookback:i,:]
        count = count + 1
        count_list.append(count)
        # 将数据透视成以日期为索引，股票代码为列，收盘价为值的表格
        price_lookback=data_lookback.pivot_table(index = 'date',columns = 'tic', values = 'close')
        # 计算价格变化的百分比，并删除任何包含NaN的行
        return_lookback = price_lookback.pct_change().dropna()
        # 计算回报率的协方差矩阵
        covs = return_lookback.cov().values 
        cov_list.append(covs)
    # 创建一个新的 DataFrame，包含日期和对应的协方差矩阵列表
    df_cov = pd.DataFrame({'date':df.date.unique()[lookback:],'cov_list':cov_list})
    # 将新的协方差矩阵 DataFrame 与原始 DataFrame 按照日期进行合并
    df = df.merge(df_cov, on='date')
    # 按日期和时间戳重新排序数据，并重置索引
    df = df.sort_values(['date','tic']).reset_index(drop=True)
    
    return df,count_list

In [ ]:
data_with_features = pd.read_csv("datasets/stock20_Feature.csv")

In [ ]:
len(data_with_features[(data_with_features.date < "2019-01-01")])/20

In [ ]:
data_with_features.tic.unique()

In [ ]:
data_with_features_covs = data_with_features.copy()
data_with_features_covs,t_List = add_cov_matrix(data_with_features_covs)

In [ ]:
len(data_with_features_covs)/20

In [ ]:
data_with_features_covs

In [ ]:
data_with_features_covs.to_csv("datasets/stock20_Features_Cov.csv",index=False)

In [ ]:
# data_with_features_covs = pd.read_csv("datasets/stock20_Features_Cov.csv")

In [ ]:
data_with_features_covs.head(1)

In [ ]:
features_data = data_with_features.columns.tolist()[7:]
features_df = data_with_features_covs[features_data]
features_df.index = data_with_features_covs['date']

In [ ]:
features_data

In [ ]:
features_df.head()

In [ ]:
len(features_df)

In [ ]:
#将 features_df转换为 NumPy 数组。
features_array = np.array(features_df)
#创建一个 MinMaxScaler 对象，用于将数据缩放到指定的范围（默认是 [0, 1]）
features_scaler = preprocessing.MinMaxScaler()
#对数据进行归一化处理，并返回归一化后的数据。
features_normalised = features_scaler.fit_transform(features_array)

In [ ]:
features_normalised.shape

In [ ]:
# rescale the features array
features_normalised = features_normalised.reshape(-1,20,10)
features_normalised.shape

#### CNN

In [ ]:
from keras.layers import Conv1D,Flatten,Reshape,MaxPooling1D
from keras.layers import Dense
from keras.layers import RepeatVector
from keras.layers import TimeDistributed
from keras.utils import plot_model
from keras.models import Model, Sequential

In [ ]:
# 定义模型
model = Sequential()
model.add(Conv1D(filters=10, kernel_size=4, activation='relu', input_shape=(20, 10)))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(RepeatVector(20))

# 使用 TimeDistributed 包装 Dense 层
# 注意，这里 Dense 层的输出单元数应该匹配你想要的特征数
# 如果你的目标是回到 10 个特征，那么可以这样设置：
model.add(TimeDistributed(Dense(10)))

# 编译模型
model.compile(optimizer='adam', loss='mse')

In [ ]:
features_normalised.shape

In [ ]:
# fit model/训练模型
model.fit(features_normalised, features_normalised, epochs=100, verbose=1)

In [ ]:
model.summary()

In [ ]:
# connect the encoder Conv1D as the output layer/将编码器 Conv1D 作为输出层连接
model_feature_CNN = Model(inputs=model.inputs, outputs=model.layers[-3].output)

In [ ]:
model_feature_CNN.summary()

In [ ]:
# get the feature vector for the input sequence
yhat_CNN = model_feature_CNN.predict(features_normalised)
print(yhat_CNN.shape)

In [ ]:
features_reduced_CNN = yhat_CNN.reshape(-1,4)

In [ ]:
features_reduced_CNN.shape

In [ ]:
features_reduced_CNN

In [ ]:
features_list_CNN = data_with_features_covs.columns.tolist()[7:-1]

In [ ]:
df_reduced_CNN = data_with_features_covs.copy()
df_reduced_CNN = df_reduced_CNN .drop(features_list_CNN, axis=1)

In [ ]:
df_reduced_CNN.head(1)

In [ ]:
# convert the reduced features to a data frame and merge with the original data frame
features_reduced_df_CNN = pd.DataFrame(features_reduced_CNN, columns=['f01','f02','f03','f04'])

In [ ]:
features_reduced_df_CNN

In [ ]:
df_reduced_CNN[['f01','f02','f03','f04']] = features_reduced_df_CNN[['f01','f02','f03','f04']]

In [ ]:
df_reduced_CNN

In [ ]:
df_reduced_CNN.to_csv("datasets/stock20_CNN.csv",index=False)

#### LSTM

In [ ]:
from numpy import array
from keras.models import Model, Sequential
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers import RepeatVector
from keras.layers import TimeDistributed
from keras.utils import plot_model
from sklearn import preprocessing

In [ ]:
# define model
model = Sequential()  #初始化一个序列模型
model.add(LSTM(4, activation='relu', input_shape=(20,10)))
model.add(RepeatVector(20))
model.add(LSTM(100, activation='relu', return_sequences=True))
model.add(TimeDistributed(Dense(10)))
model.compile(optimizer='adam', loss='mse')

In [ ]:
# fit model/训练模型
model.fit(features_normalised, features_normalised, epochs=100, verbose=1)

In [ ]:
model.summary()

In [ ]:
# connect the encoder LSTM as the output layer
model_feature = Model(inputs=model.inputs, outputs=model.layers[1].output)

In [ ]:
model_feature.summary()

In [ ]:
yhat = model_feature.predict(features_normalised)
print(yhat.shape)

In [ ]:
features_reduced = yhat.reshape(-1,4)

In [ ]:
features_reduced.shape

In [ ]:
data_with_features_covs.head()

In [ ]:
features_list = data_with_features_covs.columns.tolist()[7:-1]

In [ ]:
features_list

In [ ]:
df_reduced = data_with_features_covs.copy()
df_reduced = df_reduced .drop(features_list, axis=1)

In [ ]:
features_reduced_df = pd.DataFrame(features_reduced, columns=['f01','f02','f03','f04'])

In [ ]:
features_reduced_df

In [ ]:
df_reduced[['f01','f02','f03','f04']] = features_reduced_df[['f01','f02','f03','f04']]

In [ ]:
df_reduced.head(1)

In [ ]:
df_reduced.to_csv("datasets/stock20_LSTM.csv",index=False)

#### TC-MAC

In [ ]:
data_TC = pd.read_csv("datasets/stock20_Basic.csv")

In [ ]:
len(data_TC)

In [ ]:
data_TC.head()

In [ ]:
type(data_TC.columns)

In [ ]:
TC_new_columns = data_TC.columns[2:8]

In [ ]:
new_data_TC = data_TC[TC_new_columns]

In [ ]:
new_data_TC.shape

In [ ]:
df_test = np.reshape(new_data_TC,(-1,20,5))

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
def normalize_features(data):
    min_values = np.min(data, axis=(0, 1))
    max_values = np.max(data, axis=(0, 1))
    normalized_data = (data - min_values) / (max_values - min_values)
    return normalized_data

# 构建邻接矩阵
def build_adj_matrix(correlation_matrix):
    adj_matrix = np.abs(correlation_matrix)
    return adj_matrix

# GCN模型
class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.gc1 = nn.Linear(input_dim, hidden_dim)
        self.gc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, adj):
        x = F.relu(torch.mm(adj, self.gc1(x)))
        x = torch.mm(adj, self.gc2(x))
        return x

In [ ]:
data = df_test
normalized_data = normalize_features(data)

# 计算相关性矩阵
correlation_matrix = np.corrcoef(normalized_data[:, :, 0].T)  # 以收盘价为例
adj_matrix = build_adj_matrix(correlation_matrix)

# 转换为PyTorch张量
adj_matrix = torch.tensor(adj_matrix, dtype=torch.float32)

# 初始化GCN模型
model = GCN(input_dim=5, hidden_dim=16, output_dim=20)

# 存储每一天的输出矩阵
daily_matrices = []

# 遍历每一天的数据
for day in range(normalized_data.shape[0]):
    features = torch.tensor(normalized_data[day, :, :], dtype=torch.float32)  # 取每一天的数据
    output = model(features, adj_matrix)
    # 归一化输出
    normalized_output = (output - torch.min(output)) / (torch.max(output) - torch.min(output))
    daily_matrices.append(normalized_output.detach().numpy())  # 使用.detach().numpy()避免错误
# 将所有天的矩阵转换为一个三维数组
daily_matrices = np.array(daily_matrices)
print("Shape of daily matrices:", daily_matrices.shape) 
print("Example of first day's matrix:\n", daily_matrices[0])

In [ ]:
list_con  = [] 
for i in range(0,daily_matrices.shape[0]):
    list_con.append(daily_matrices[i])

In [ ]:
df_GCN = pd.DataFrame({'date':data_TC.date.unique(),'cov_list':list_con})

In [ ]:
data_TC = data_TC.merge(df_GCN,on="date")
data_TC = data_TC.sort_values(['date','tic']).reset_index(drop=True)

In [ ]:
data_TC.head()

In [ ]:
from sklearn import preprocessing
from keras.models import Sequential,Model
from keras.layers import GRU, RepeatVector, TimeDistributed, Dense
from keras.utils import plot_model

In [ ]:
new_data_TC.shape

In [ ]:
df_array = np.array(new_data_TC)
features_scaler = preprocessing.MinMaxScaler()
#对数据进行归一化处理，并返回归一化后的数据。
features_normalised = features_scaler.fit_transform(df_array)

In [ ]:
# rescale the features array
features_normalised = features_normalised.reshape(-1,20,5)
features_normalised.shape

In [ ]:
model = Sequential()
model.add(GRU(4, activation='relu', input_shape=(20, 5), return_sequences=False))
model.add(RepeatVector(20))
model.add(GRU(100, activation='relu', return_sequences=True))
model.add(TimeDistributed(Dense(5)))
model.compile(optimizer='adam', loss='mse')

In [ ]:
model.fit(features_normalised, features_normalised, epochs=100, verbose=1)

In [ ]:
features_normalised

In [ ]:
model.summary()

In [ ]:

model_feature = Model(inputs=model.inputs, outputs=model.layers[1].output)

In [ ]:
model_feature.summary()

In [ ]:
yhat = model_feature.predict(features_normalised)
print(yhat.shape)

In [ ]:
features_reduced = yhat.reshape(-1,4)

In [ ]:
len(features_reduced)

In [ ]:
features_reduced

In [ ]:
df_reduced = data_TC.copy()

In [ ]:
df_reduced.head()

In [ ]:
features_reduced_df = pd.DataFrame(features_reduced, columns=['f01','f02','f03','f04'])

In [ ]:
features_reduced_df

In [ ]:
df_reduced[['f01','f02','f03','f04']] = features_reduced_df[['f01','f02','f03','f04']]

In [ ]:
df_reduced.head(11)

In [ ]:
df_reduced.to_csv("datasets/stock20_TC-MAC.csv",index=False)

In [ ]:
df_reduced.tic.unique()

### 数据分割

In [ ]:
from finrl.meta.preprocessor.preprocessors import data_split

In [ ]:
LSTM_data = pd.read_csv("datasets/stock20_LSTM.csv")
CNN_data = pd.read_csv("datasets/stock20_CNN.csv")
TC_MAC_data = pd.read_csv("datasets/stock20_TC-MAC.csv")

In [ ]:
LSTM_data

In [ ]:
LSTM_data.columns=["date","tic","open","close","high","low","volume","cov_list","f1","f2","f3","f4"]

In [ ]:
len(LSTM_data.date.unique())

In [ ]:
CNN_data.columns=["date","tic","open","close","high","low","volume","cov_list","f1","f2","f3","f4"]

In [ ]:
len(CNN_data.date.unique())

In [ ]:
TC_MAC_data = TC_MAC_data[TC_MAC_data.date >= "2019-01-01"]

In [ ]:
len(TC_MAC_data.date.unique())

In [ ]:
TC_MAC_data.columns=["date","tic","open","close","high","low","volume","cov_list","f1","f2","f3","f4"]

In [ ]:
len(TC_MAC_data.date.unique())

In [ ]:
train_start_date = "2019-01-01"
train_end_date = "2022-12-31"
test_start_date ="2023-01-01"
test_end_date = "2023-12-31"

In [ ]:
train_data_LSTM = data_split(LSTM_data, train_start_date, train_end_date)
# verify_data_LSTM = data_split(LSTM_data, verify_start_date, verify_end_date)
test_data_LSTM = data_split(LSTM_data, test_start_date, test_end_date)

train_data_CNN = data_split(CNN_data, train_start_date, train_end_date)
# verify_data_CNN = data_split(CNN_data, verify_start_date, verify_end_date)
test_data_CNN = data_split(CNN_data, test_start_date, test_end_date)

train_data_TC = data_split(TC_MAC_data, train_start_date, train_end_date)
# verify_data_TC = data_split(TC_MAC_data, verify_start_date, verify_end_date)
test_data_TC = data_split(TC_MAC_data, test_start_date, test_end_date)


In [ ]:
print(len(train_data_LSTM))
# print(len(verify_data_LSTM))
print(len(test_data_LSTM))

print(len(train_data_CNN))
# print(len(verify_data_CNN))
print(len(test_data_CNN))

print(len(train_data_TC))
# print(len(verify_data_TC))
print(len(test_data_TC))

In [ ]:
train_data_LSTM.to_csv("datasets/01_train_LSTM.csv",index=False)
# verify_data_LSTM.to_csv("datasets/test2/01_verify_LSTM.csv",index=False)
test_data_LSTM.to_csv("datasets/01_test_LSTM.csv",index=False)

In [ ]:
train_data_CNN.to_csv("datasets/02_train_CNN.csv",index=False)
# verify_data_CNN.to_csv("datasets/test2/02_verify_CNN.csv",index=False)
test_data_CNN.to_csv("datasets/02_test_CNN.csv",index=False)

In [ ]:
train_data_TC.to_csv("datasets/03_train_TC.csv",index=False)
# verify_data_TC.to_csv("datasets/test2/03_verify_TC.csv",index=False)
test_data_TC.to_csv("datasets/03_test_TC.csv",index=False)